# XGBoost Optimization — Feature Selection, Tuning & Calibration

**Prerequisites:**
- Run `0 - build_features.ipynb` to generate `data/latest_features.jsonl`

**Goal:** Find the optimal XGBoost configuration for live trading:
1. Feature importance ranking → optimal feature count
2. Hyperparameter grid search with early stopping
3. Probability calibration (isotonic regression)
4. Forward-test on unseen candles from DB
5. Head-to-head comparison vs LR and RF
6. Save `data/optimal_features_xgb.json`


In [ ]:
import json
import random
import sqlite3
from datetime import UTC, datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, brier_score_loss, log_loss
from sklearn.model_selection import ParameterGrid
from sklearn.preprocessing import StandardScaler
from technicals import CandleRecord, IndicatorSnapshot, compute_all
from tqdm import tqdm

random.seed(42)
np.random.seed(42)

FEATURES_PATH = Path("../data/latest_features.jsonl")
LR_FEATURES_PATH = Path("../data/optimal_features_lr.json")
RF_FEATURES_PATH = Path("../data/optimal_features_rf.json")
DB_PATH = Path("../data/collection.db")
MAX_BID = 0.85
WARM_UP = 21

## 1. Load data and train/val split

In [ ]:
rows = []
with open(FEATURES_PATH) as f:
    for line in f:
        rows.append(json.loads(line))

df = pd.DataFrame(rows)
df["target"] = (df["outcome"] == "UP").astype(int)

NON_FEAT = {
    "candle_id",
    "session",
    "timestamp",
    "elapsed_pct",
    "btc_price",
    "up_best_bid",
    "up_best_ask",
    "up_bid_depth",
    "up_ask_depth",
    "down_best_bid",
    "down_best_ask",
    "down_bid_depth",
    "down_ask_depth",
    "market_volume",
    "outcome",
    "target",
}
all_feat_cols = sorted([c for c in df.columns if c not in NON_FEAT])
df[all_feat_cols] = df[all_feat_cols].fillna(0.0)

candle_ids = df["candle_id"].unique()
split_idx = int(len(candle_ids) * 0.8)
train_ids = set(candle_ids[:split_idx])
val_ids = set(candle_ids[split_idx:])

df_train = df[df["candle_id"].isin(train_ids)]
df_val = df[df["candle_id"].isin(val_ids)]

scaler = StandardScaler()
X_train_all = scaler.fit_transform(df_train[all_feat_cols].values)
y_train = df_train["target"].values
X_val_all = scaler.transform(df_val[all_feat_cols].values)
y_val = df_val["target"].values

print(f"Features: {len(all_feat_cols)}")
print(f"Train: {len(df_train):,} rows, {df_train['candle_id'].nunique()} candles")
print(f"Val:   {len(df_val):,} rows, {df_val['candle_id'].nunique()} candles")

## 2. Feature importance ranking with XGBoost

In [ ]:
# Train baseline XGBoost on all features to get importance ranking
base_xgb = xgb.XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=10,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss",
    n_jobs=-1,
)
base_xgb.fit(X_train_all, y_train)

importances = base_xgb.feature_importances_
feat_imp = sorted(zip(all_feat_cols, importances, strict=False), key=lambda x: -x[1])

print(f"{'Feature':<35} {'Importance':>10}")
print("-" * 47)
for name, imp in feat_imp[:30]:
    bar = "█" * int(imp / feat_imp[0][1] * 30)
    print(f"  {name:<33} {imp:>10.4f}  {bar}")

In [ ]:
names = [f[0] for f in feat_imp]
imps = [f[1] for f in feat_imp]

fig, ax = plt.subplots(figsize=(10, 14))
colors = ["#2ecc71" if i < 20 else "#95a5a6" for i in range(len(names))]
ax.barh(range(len(names)), imps, color=colors, edgecolor="white")
ax.set_yticks(range(len(names)))
ax.set_yticklabels(names, fontsize=8)
ax.invert_yaxis()
ax.set_xlabel("Feature Importance (gain)")
ax.set_title("XGBoost Feature Importance Ranking")
plt.tight_layout()
plt.show()

## 3. Find optimal feature count

In [ ]:
ranked_features = [name for name, _ in feat_imp]
ranked_indices = [all_feat_cols.index(name) for name in ranked_features]

top_n_results = []
print(f"{'Top N':<8} {'Snap Acc':>9} {'Brier':>8}")
print("-" * 28)

for n in list(range(5, len(all_feat_cols) + 1, 5)) + [len(all_feat_cols)]:
    idx = ranked_indices[:n]
    model = xgb.XGBClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=6,
        min_child_weight=10,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric="logloss",
        n_jobs=-1,
    )
    model.fit(X_train_all[:, idx], y_train)
    probs = model.predict_proba(X_val_all[:, idx])[:, 1]
    snap_acc = accuracy_score(y_val, (probs >= 0.5).astype(int))
    brier = brier_score_loss(y_val, probs)
    top_n_results.append((n, snap_acc, brier))
    print(f"  {n:<6} {snap_acc * 100:>8.1f}% {brier:>8.4f}")

# Best N by snapshot accuracy
best_n, best_snap, best_brier = max(top_n_results, key=lambda x: x[1])
print(f"\nBest: top {best_n} features → {best_snap * 100:.1f}% accuracy, Brier={best_brier:.4f}")

optimal_names = ranked_features[:best_n]
optimal_indices = ranked_indices[:best_n]
print(f"Optimal features: {optimal_names}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ns = [r[0] for r in top_n_results]
snap_accs = [r[1] * 100 for r in top_n_results]
briers = [r[2] for r in top_n_results]

axes[0].plot(ns, snap_accs, "o-", color="steelblue")
axes[0].axvline(best_n, color="green", linestyle="--", alpha=0.5, label=f"optimal N={best_n}")
axes[0].set_xlabel("Number of Features")
axes[0].set_ylabel("Snapshot Accuracy (%)")
axes[0].set_title("Accuracy vs Feature Count")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(ns, briers, "o-", color="darkorange")
axes[1].axvline(best_n, color="green", linestyle="--", alpha=0.5, label=f"optimal N={best_n}")
axes[1].set_xlabel("Number of Features")
axes[1].set_ylabel("Brier Score (lower = better)")
axes[1].set_title("Calibration vs Feature Count")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle("XGBoost Feature Selection", fontsize=13)
plt.tight_layout()
plt.show()

## 4. Hyperparameter tuning with early stopping

In [ ]:
X_train_opt = X_train_all[:, optimal_indices]
X_val_opt = X_val_all[:, optimal_indices]

param_grid = {
    "learning_rate": [0.01, 0.03, 0.05],
    "max_depth": [3, 4, 6],
    "min_child_weight": [10, 15, 20],
    "subsample": [0.7, 0.8],
    "colsample_bytree": [0.6, 0.7, 0.8],
}

grid = list(ParameterGrid(param_grid))
print(f"Testing {len(grid)} hyperparameter combinations...")

results = []
for _i, params in enumerate(tqdm(grid, desc="Tuning")):
    model = xgb.XGBClassifier(
        n_estimators=1000,
        early_stopping_rounds=30,
        random_state=42,
        eval_metric="logloss",
        n_jobs=-1,
        **params,
    )
    model.fit(
        X_train_opt,
        y_train,
        eval_set=[(X_val_opt, y_val)],
        verbose=False,
    )
    probs = model.predict_proba(X_val_opt)[:, 1]
    acc = accuracy_score(y_val, (probs >= 0.5).astype(int))
    brier = brier_score_loss(y_val, probs)
    n_trees = model.best_iteration + 1
    results.append({**params, "accuracy": acc, "brier": brier, "n_estimators": n_trees})

results_df = pd.DataFrame(results).sort_values("accuracy", ascending=False)
print("\nTop 10 configurations:")
print(results_df.head(10).to_string(index=False))

best_params = results_df.iloc[0].to_dict()
best_acc = best_params.pop("accuracy")
best_brier_tuned = best_params.pop("brier")
best_n_est = int(best_params.pop("n_estimators"))
# Cast types: max_depth, min_child_weight, n_estimators must be int for XGBoost
INT_PARAMS = {"max_depth", "min_child_weight", "n_estimators"}
best_params = {k: int(v) if k in INT_PARAMS else float(v) for k, v in best_params.items()}
best_params["n_estimators"] = best_n_est

print(f"\nBest: accuracy={best_acc * 100:.1f}%, Brier={best_brier_tuned:.4f}")
print(f"Params: {best_params}")

## 5. Probability calibration

In [ ]:
# Train the best model
best_xgb = xgb.XGBClassifier(
    random_state=42,
    eval_metric="logloss",
    n_jobs=-1,
    **best_params,
)
best_xgb.fit(X_train_opt, y_train)

# Raw probabilities
raw_probs = best_xgb.predict_proba(X_val_opt)[:, 1]

# Calibrate with isotonic regression (cv=3 on training data)
cal_xgb = CalibratedClassifierCV(best_xgb, method="isotonic", cv=3)
cal_xgb.fit(X_train_opt, y_train)
cal_probs = cal_xgb.predict_proba(X_val_opt)[:, 1]

# Compare
raw_acc = accuracy_score(y_val, (raw_probs >= 0.5).astype(int))
cal_acc = accuracy_score(y_val, (cal_probs >= 0.5).astype(int))
raw_brier = brier_score_loss(y_val, raw_probs)
cal_brier = brier_score_loss(y_val, cal_probs)
raw_logloss = log_loss(y_val, raw_probs)
cal_logloss = log_loss(y_val, cal_probs)

print(f"{'Metric':<20} {'Raw':>10} {'Calibrated':>12}")
print("-" * 44)
print(f"{'Accuracy':<20} {raw_acc * 100:>9.1f}% {cal_acc * 100:>11.1f}%")
print(f"{'Brier Score':<20} {raw_brier:>10.4f} {cal_brier:>12.4f}")
print(f"{'Log Loss':<20} {raw_logloss:>10.4f} {cal_logloss:>12.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, probs, label, color in [
    (axes[0], raw_probs, "Raw XGBoost", "steelblue"),
    (axes[1], cal_probs, "Calibrated XGBoost", "#2ecc71"),
]:
    fraction_pos, mean_predicted = calibration_curve(y_val, probs, n_bins=10)
    ax.plot(mean_predicted, fraction_pos, "o-", color=color, label=label)
    ax.plot([0, 1], [0, 1], "k--", alpha=0.3, label="perfectly calibrated")
    ax.set_xlabel("Mean Predicted Probability")
    ax.set_ylabel("Fraction of Positives")
    brier = brier_score_loss(y_val, probs)
    ax.set_title(f"{label} (Brier={brier:.4f})")
    ax.legend()
    ax.grid(alpha=0.3)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

plt.suptitle("Reliability Diagram — Raw vs Calibrated", fontsize=13)
plt.tight_layout()
plt.show()

### Export optimal XGBoost features

Save to `data/optimal_features_xgb.json` for all downstream notebooks.

In [ ]:
config = {
    "model": "xgboost",
    "features": optimal_names,
    "n_features": best_n,
    "accuracy": round(best_acc, 4),
    "selection_method": "importance_ranking",
    "hyperparameters": best_params,
    "calibrated": True,
    "source": "data/latest_features.jsonl",
    "created_at": datetime.now(UTC).isoformat(),
}

out_path = Path("../data/optimal_features_xgb.json")
with open(out_path, "w") as f:
    json.dump(config, f, indent=2)

print(f"Saved {config['n_features']} XGBoost features to {out_path}")
print(f"Accuracy: {config['accuracy'] * 100:.2f}%")
print(f"Hyperparameters: {config['hyperparameters']}")
print(f"Features: {config['features']}")

## 6. Forward-test on newest DB candles

In [ ]:
# Train on ALL JSONL data with optimal features + best params
scaler_final = StandardScaler()
X_all = scaler_final.fit_transform(df[optimal_names].values)
y_all = df["target"].values

xgb_final = xgb.XGBClassifier(
    random_state=42,
    eval_metric="logloss",
    n_jobs=-1,
    **best_params,
)
xgb_final.fit(X_all, y_all)

# Calibrate on a holdout fold
cal_final = CalibratedClassifierCV(xgb_final, method="isotonic", cv=3)
cal_final.fit(X_all, y_all)

max_train_ts = df["timestamp"].max()
print(f"Trained on {df['candle_id'].nunique()} candles ({len(df):,} rows)")
print(f"Newest training timestamp: {max_train_ts}")

In [ ]:
conn = sqlite3.connect(str(DB_PATH))
candles_df = pd.read_sql(
    f"SELECT * FROM candles WHERE start_time > {max_train_ts} ORDER BY start_time",
    conn,
)
snaps_df = (
    pd.read_sql(
        "SELECT * FROM snapshots WHERE candle_id IN ({}) ORDER BY candle_id, timestamp".format(
            ",".join(f"'{cid}'" for cid in candles_df["candle_id"])
        ),
        conn,
    )
    if len(candles_df) > 0
    else pd.DataFrame()
)
prior_candles_df = pd.read_sql(
    f"SELECT * FROM candles WHERE start_time <= {max_train_ts} ORDER BY start_time DESC LIMIT {WARM_UP}",
    conn,
)
conn.close()

prior_candles_df = prior_candles_df.sort_values("start_time")
prior_candles = []
for _, cr in prior_candles_df.iterrows():
    prior_candles.append(
        CandleRecord(
            candle_id=cr["candle_id"],
            start_time=cr["start_time"],
            end_time=cr["end_time"],
            open=cr["open"],
            high=cr["high"],
            low=cr["low"],
            close=cr["close"],
            volume=cr["volume"],
            outcome=cr["outcome"],
            final_ret=cr["final_ret"],
        )
    )

all_rows = []
for _, cr in tqdm(candles_df.iterrows(), total=len(candles_df), desc="Computing features"):
    cid = cr["candle_id"]
    candle = CandleRecord(
        candle_id=cid,
        start_time=cr["start_time"],
        end_time=cr["end_time"],
        open=cr["open"],
        high=cr["high"],
        low=cr["low"],
        close=cr["close"],
        volume=cr["volume"],
        outcome=cr["outcome"],
        final_ret=cr["final_ret"],
    )
    snap_rows = snaps_df[snaps_df["candle_id"] == cid]
    if len(snap_rows) < 5:
        prior_candles.append(candle)
        continue
    snapshots = []
    for _, s in snap_rows.iterrows():
        ob = json.loads(s["orderbook_json"])
        snapshots.append(
            IndicatorSnapshot(
                candle_id=cid,
                timestamp=s["timestamp"],
                elapsed_pct=s["elapsed_pct"],
                btc_price=s["btc_price"],
                btc_bid=s["btc_bid"],
                btc_ask=s["btc_ask"],
                up_bids=[ob["up_bids"][0]] if ob.get("up_bids") else [],
                up_asks=[ob["up_asks"][0]] if ob.get("up_asks") else [],
                down_bids=[ob["down_bids"][0]] if ob.get("down_bids") else [],
                down_asks=[ob["down_asks"][0]] if ob.get("down_asks") else [],
                market_volume=s["market_volume"],
            )
        )
    for si in range(len(snapshots)):
        indicators = compute_all(prior_candles, candle.open, snapshots[: si + 1])
        snap = snapshots[si]
        row = {
            "candle_id": cid,
            "timestamp": snap.timestamp,
            "elapsed_pct": snap.elapsed_pct,
            "btc_price": snap.btc_price,
            "up_best_ask": snap.up_asks[0][0] if snap.up_asks else None,
            "down_best_ask": snap.down_asks[0][0] if snap.down_asks else None,
            **indicators,
            "outcome": candle.outcome,
        }
        all_rows.append(row)
    prior_candles.append(candle)

df_eval = pd.DataFrame(all_rows)
df_eval["target"] = (df_eval["outcome"] == "UP").astype(int)
for col in all_feat_cols:
    if col not in df_eval.columns:
        df_eval[col] = 0.0
df_eval[all_feat_cols] = df_eval[all_feat_cols].fillna(0.0)

print(f"\nNew candles: {df_eval['candle_id'].nunique()}")
print(f"Rows: {len(df_eval):,}")

## 7. Build per-candle predictions — XGB, LR, RF

In [ ]:
# Also train LR and RF for comparison
with open(LR_FEATURES_PATH) as _f:
    lr_feat_cols = sorted(json.load(_f)["features"])
with open(RF_FEATURES_PATH) as _f:
    rf_feat_cols = json.load(_f)["features"]

# LR
lr_scaler = StandardScaler()
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(lr_scaler.fit_transform(df[lr_feat_cols].values), y_all)

# RF
rf_scaler = StandardScaler()
rf_model = RandomForestClassifier(n_estimators=200, max_depth=15, min_samples_leaf=20, random_state=42, n_jobs=-1)
rf_model.fit(rf_scaler.fit_transform(df[rf_feat_cols].values), y_all)

print(f"LR: {len(lr_feat_cols)} features | RF: {len(rf_feat_cols)} features | XGB: {len(optimal_names)} features")

# Build predictions for all 3 models
all_cd = []
for cid in df_eval["candle_id"].unique():
    snap_rows = df_eval[df_eval["candle_id"] == cid].sort_values("timestamp")
    if len(snap_rows) < 5:
        continue
    truth = int(snap_rows["target"].iloc[0])

    X_xgb = scaler_final.transform(snap_rows[optimal_names].values)
    xgb_probs = cal_final.predict_proba(X_xgb)[:, 1]

    X_lr = lr_scaler.transform(snap_rows[lr_feat_cols].values)
    lr_probs = lr_model.predict_proba(X_lr)[:, 1]

    X_rf = rf_scaler.transform(snap_rows[rf_feat_cols].values)
    rf_probs = rf_model.predict_proba(X_rf)[:, 1]

    up_asks = snap_rows["up_best_ask"].values
    down_asks = snap_rows["down_best_ask"].values
    elapsed = snap_rows["elapsed_pct"].values

    sd = [
        {
            "tick": i,
            "elapsed_pct": elapsed[i],
            "xgb_pred": int(xgb_probs[i] >= 0.5),
            "xgb_prob": float(xgb_probs[i]),
            "lr_pred": int(lr_probs[i] >= 0.5),
            "lr_prob": float(lr_probs[i]),
            "rf_pred": int(rf_probs[i] >= 0.5),
            "rf_prob": float(rf_probs[i]),
            "up_ask": up_asks[i],
            "down_ask": down_asks[i],
        }
        for i in range(len(snap_rows))
    ]
    all_cd.append({"candle_id": cid, "truth": truth, "snapshots": sd})

print(f"Built predictions for {len(all_cd)} candles")

## 8. Scaling-in strategies + confidence filtering

In [ ]:
def run_scaling(name, entry_points, model_key="xgb", bet_per_entry=10.0, min_confidence=0.0):
    pred_key = f"{model_key}_pred"
    prob_key = f"{model_key}_prob"
    bal = 1000.0
    history = [bal]
    total_bets, total_wins, candles_entered = 0, 0, 0

    for cd in all_cd:
        sd = cd["snapshots"]
        truth = cd["truth"]
        entries = []
        first_direction = None

        for min_e, n_c in entry_points:
            for i in range(max(n_c - 1, 0), len(sd)):
                if sd[i]["elapsed_pct"] < min_e:
                    continue
                if any(i <= prev_tick for prev_tick, _, _ in entries):
                    continue
                if n_c > 1 and not all(sd[i - j][pred_key] == sd[i][pred_key] for j in range(n_c)):
                    continue
                confidence = max(sd[i][prob_key], 1.0 - sd[i][prob_key])
                if confidence < min_confidence:
                    continue
                direction = sd[i][pred_key]
                if first_direction is None:
                    first_direction = direction
                elif direction != first_direction:
                    break
                ask = sd[i]["up_ask"] if direction == 1 else sd[i]["down_ask"]
                if ask is None or not np.isfinite(ask) or ask <= 0 or ask >= MAX_BID:
                    continue
                entries.append((i, direction, ask))
                break

        if not entries:
            continue
        candles_entered += 1
        for _, direction, ask in entries:
            if bal < bet_per_entry:
                break
            total_bets += 1
            if direction == truth:
                bal += (bet_per_entry / ask) * (1.0 - ask)
                total_wins += 1
            else:
                bal -= bet_per_entry
        history.append(bal)

    wr = total_wins / total_bets if total_bets > 0 else 0
    max_dd = (
        max(
            (peak_v - h) / peak_v
            for peak_v, h in zip([max(history[: i + 1]) for i in range(len(history))], history, strict=False)
        )
        if len(history) > 1
        else 0
    )
    return {
        "name": name,
        "balance": bal,
        "history": history,
        "total_bets": total_bets,
        "candles_entered": candles_entered,
        "wins": total_wins,
        "win_rate": wr,
        "return": (bal - 1000) / 10,
        "max_dd": max_dd,
    }

In [ ]:
strategies = [
    # XGBoost strategies
    ("XGB 1x e5%", [(0.05, 3)], "xgb", 0.0),
    ("XGB 1x e5% conf>0.6", [(0.05, 3)], "xgb", 0.6),
    ("XGB 1x e5% conf>0.7", [(0.05, 3)], "xgb", 0.7),
    ("XGB 2x e5%+e50%", [(0.05, 3), (0.50, 3)], "xgb", 0.0),
    ("XGB 2x e5%+e50% conf>0.6", [(0.05, 3), (0.50, 3)], "xgb", 0.6),
    ("XGB 2x e5%+e50% conf>0.7", [(0.05, 3), (0.50, 3)], "xgb", 0.7),
    # LR baseline
    ("LR 2x e5%+e50% conf>0.7", [(0.05, 3), (0.50, 3)], "lr", 0.7),
    # RF baseline
    ("RF 2x e5%+e50% conf>0.7", [(0.05, 3), (0.50, 3)], "rf", 0.7),
]

results = []
print(f"{'Strategy':<30} {'Bets':>5} {'WR':>6} {'Balance':>10} {'Return':>8} {'MaxDD':>7}")
print("-" * 72)

for name, eps, model_key, conf in strategies:
    r = run_scaling(name, eps, model_key=model_key, min_confidence=conf)
    results.append(r)
    print(
        f"{r['name']:<30} {r['total_bets']:>5} "
        f"{r['win_rate'] * 100:>5.1f}% ${r['balance']:>9.2f} {r['return']:>+7.1f}% {r['max_dd'] * 100:>6.1f}%"
    )

## 9. Head-to-head equity curves

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 10))

colors = {"XGB": "#e67e22", "LR": "#3498db", "RF": "#e74c3c"}
for r in results:
    prefix = r["name"].split(" ")[0]
    axes[0].plot(r["history"], color=colors.get(prefix, "gray"), alpha=0.4, linewidth=1)

for prefix, color in colors.items():
    axes[0].plot([], [], color=color, label=prefix, linewidth=2)
axes[0].axhline(1000, color="gray", linestyle="--", alpha=0.3)
axes[0].set_xlabel("Candle #")
axes[0].set_ylabel("Balance ($)")
axes[0].set_title("All Strategies — XGB (orange) vs LR (blue) vs RF (red)")
axes[0].legend()
axes[0].grid(alpha=0.3)

# Best of each
best = {}
for r in results:
    prefix = r["name"].split(" ")[0]
    if prefix not in best or r["balance"] > best[prefix]["balance"]:
        best[prefix] = r

for prefix in ["XGB", "LR", "RF"]:
    if prefix in best:
        r = best[prefix]
        axes[1].plot(
            r["history"],
            color=colors[prefix],
            linewidth=2.5,
            label=f"{r['name']} → ${r['balance']:,.0f} ({r['return']:+.1f}%, WR={r['win_rate'] * 100:.0f}%)",
        )

axes[1].axhline(1000, color="gray", linestyle="--", alpha=0.3)
axes[1].set_xlabel("Candle #")
axes[1].set_ylabel("Balance ($)")
axes[1].set_title("Best XGB vs Best LR vs Best RF")
axes[1].legend(fontsize=10)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 10. Summary

In [ ]:
print("=" * 72)
print("XGBOOST OPTIMIZATION — FORWARD TEST SUMMARY")
print(f"Candles: {len(all_cd)} (unseen during training)")
print(f"Optimal features: {best_n} (from {len(all_feat_cols)} total)")
print(f"Best hyperparameters: {best_params}")
print("Calibrated: Yes (isotonic regression)")
print("=" * 72)

print(f"\n{'':30} {'XGB best':>20} {'LR best':>20} {'RF best':>20}")
print("-" * 92)
for metric, key in [
    ("Strategy", "name"),
    ("Win Rate", "win_rate"),
    ("Return", "return"),
    ("Max Drawdown", "max_dd"),
    ("Final Balance", "balance"),
    ("Total Bets", "total_bets"),
]:
    vals = []
    for prefix in ["XGB", "LR", "RF"]:
        r = best.get(prefix, {})
        v = r.get(key, "N/A")
        if key == "win_rate":
            vals.append(f"{v * 100:.1f}%" if isinstance(v, float) else str(v))
        elif key == "return":
            vals.append(f"{v:+.1f}%" if isinstance(v, int | float) else str(v))
        elif key == "max_dd":
            vals.append(f"{v * 100:.1f}%" if isinstance(v, float) else str(v))
        elif key == "balance":
            vals.append(f"${v:,.2f}" if isinstance(v, float) else str(v))
        else:
            vals.append(str(v))
    print(f"{metric:<30} {vals[0]:>20} {vals[1]:>20} {vals[2]:>20}")

print("\nNote: Re-run after collecting more data for robust comparison.")

## 11. Conclusion

XGBoost optimized with feature selection, hyperparameter tuning, and probability calibration.

- Optimal features saved to `data/optimal_features_xgb.json`
- Run `15 - export_xgb_model.ipynb` to export the final model for live trading
- Re-run after collecting more data to refine the configuration